### EDA

### load dataset

In [1]:
import pandas as pd
data=pd.read_csv('../data//raw/dataset.csv')
print(data)

                                                 subject  \
0      Unvorhergesehener Absturz der Datenanalyse-Pla...   
1                               Customer Support Inquiry   
2                          Data Analytics for Investment   
3                     Krankenhaus-Dienstleistung-Problem   
4                                               Security   
...                                                  ...   
19995     Assistance Needed for IFTTT Docker Integration   
19996        Bitten um Unterstützung bei der Integration   
19997                                                NaN   
19998            Hilfe bei digitalen Strategie-Problemen   
19999  Optimierung Ihrer Datenanalyse-Plattform erlei...   

                                                    body  \
0      Die Datenanalyse-Plattform brach unerwartet ab...   
1      Seeking information on digital strategies that...   
2      I am contacting you to request information on ...   
3      Ein Medien-Daten-Sperrverhalten 

### Inspection du Data : 

In [2]:
data.shape

(20000, 15)

In [3]:
data.columns

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
      dtype='str')

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   subject   18539 non-null  str  
 1   body      19998 non-null  str  
 2   answer    19996 non-null  str  
 3   type      20000 non-null  str  
 4   queue     20000 non-null  str  
 5   priority  20000 non-null  str  
 6   language  20000 non-null  str  
 7   tag_1     20000 non-null  str  
 8   tag_2     19954 non-null  str  
 9   tag_3     19905 non-null  str  
 10  tag_4     18461 non-null  str  
 11  tag_5     13091 non-null  str  
 12  tag_6     7351 non-null   str  
 13  tag_7     3928 non-null   str  
 14  tag_8     1907 non-null   str  
dtypes: str(15)
memory usage: 19.9 MB


In [5]:
print(data.isnull().sum() / len(data) * 100)

subject      7.305
body         0.010
answer       0.020
type         0.000
queue        0.000
priority     0.000
language     0.000
tag_1        0.000
tag_2        0.230
tag_3        0.475
tag_4        7.695
tag_5       34.545
tag_6       63.245
tag_7       80.360
tag_8       90.465
dtype: float64


In [6]:
# Gérer les textes principaux
data['subject'] = data['subject'].fillna('')
data['body'] = data['body'].fillna('')
data['answer'] = data['answer'].fillna('')

# Créer le texte combiné pour NLP
data['text'] = data['subject'] + ' ' + data['body']

# Gérer les tags utiles (< 8% nulls)
for col in ['tag_2', 'tag_3', 'tag_4']:
    data[col] = data[col].fillna('')

# SUPPRIMER les colonnes trop vides (> 30% nulls)
data = data.drop(columns=['tag_5', 'tag_6', 'tag_7', 'tag_8'])

# 6. Vérification finale
print("\n Vérification après nettoyage :")
print(data.isnull().sum())
print(f"\nNombre de colonnes conservées : {data.shape[1]}")
print(f"Colonnes finales : {data.columns.tolist()}")


 Vérification après nettoyage :
subject     0
body        0
answer      0
type        0
queue       0
priority    0
language    0
tag_1       0
tag_2       0
tag_3       0
tag_4       0
text        0
dtype: int64

Nombre de colonnes conservées : 12
Colonnes finales : ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'text']


In [7]:
data=data.drop(columns=['subject', 'body'])

In [8]:
data['language'].value_counts()

language
en    11923
de     8077
Name: count, dtype: int64

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   answer    20000 non-null  str  
 1   type      20000 non-null  str  
 2   queue     20000 non-null  str  
 3   priority  20000 non-null  str  
 4   language  20000 non-null  str  
 5   tag_1     20000 non-null  str  
 6   tag_2     20000 non-null  str  
 7   tag_3     20000 non-null  str  
 8   tag_4     20000 non-null  str  
 9   text      20000 non-null  str  
dtypes: str(10)
memory usage: 18.9 MB


In [10]:
data['type'].value_counts()

type
Incident    7978
Request     5763
Problem     4184
Change      2075
Name: count, dtype: int64

In [11]:
data.describe()

,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,text
count,20000,20000,20000,20000,20000,20000,20000,20000,20000,20000
unique,19997,4,10,3,2,148,205,345,482,20000
top,,Incident,Technical Support,medium,en,Technical,Performance,IT,Tech Support,Unvorhergesehener Absturz der Datenanalyse-Pla...
freq,4,7978,5824,8144,11923,5034,2795,3309,3436,1


In [12]:
# find duplicates
data.duplicated().sum()

np.int64(0)

In [13]:
d=len(data['text'].max())
print(d)

464


### imports

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
import ssl

# importation des ressources NLTK   
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print(" Ressources NLTK OK\n")
except Exception as e:
    print(f" Erreur téléchargement NLTK: {e}")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

 Ressources NLTK OK



## Nettoyage NLP :

### normalisation et suppression ponctuation


In [15]:
def clean_text(text):
    """Nettoyage NLP de base"""
    if pd.isna(text) or text == '':
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Suppression ponctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 3. Suppression espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

data['text_cleaned'] = data['text'].apply(clean_text)

### tokenisation and suppression des stopwords

In [16]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def tokenize_multilingual(text):
    """Tokenisation avec stopwords anglais ET allemand"""
    if not text or text == '':
        return []
    
    try:
        # Tokenisation
        tokens = word_tokenize(text)
        
        # Combiner stopwords des 2 langues
        stop_en = set(stopwords.words('english'))
        stop_de = set(stopwords.words('german'))
        all_stopwords = stop_en.union(stop_de)
        
        # Filtrer
        tokens = [
            word for word in tokens 
            if word not in all_stopwords and len(word) > 2
        ]
        
        return tokens
    
    except Exception as e:
        print(f" Erreur: {e}")
        return text.split()

data['tokens']=data['text'].apply(tokenize_multilingual)


In [17]:
data.columns

Index(['answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2',
       'tag_3', 'tag_4', 'text', 'text_cleaned', 'tokens'],
      dtype='str')